<CENTER><img src="images/logos.png" style="width:50%"></CENTER>

# Higgs $\to 4l$ with a Neural Net

The following analysis will aim to sort events containing a Higgs boson decaying into 4 leptons from similar looking background events. 

In this notebook, `leptons' refers to electrons or muons (sorry taus!). We will introduce a neural net to classify events as either "background", or "signal" Higgs events. 

This notebook is inspired by this article: https://atlas.cern/updates/briefing/exploring-higgs-discovery-channels and corresponding paper: https://atlas.web.cern.ch/Atlas/GROUPS/PHYSICS/PAPERS/HIGG-2018-29/

**Contents:** <a name="c"></a>
- [Initial setup](#0.)
- [Reading in ROOT files](#1.)
- [Selecting events and preparing data](#2.)
- [Training a Neural Net](#3.)
- [Running on ATLAS data](#4.)
- ['Do your own project' ideas](#5.)
---

## 0. Initial setup <a name="0."></a>

### Importing libraries

Since this is a new notebook, we'll need to import the usual python libraries

In [ ]:
import uproot
import pandas as pd
from TLorentzVector import TLorentzVector
import numpy as np
import matplotlib.pyplot as plt
import time
from FileRetriever import print_files as print_files

[Return to contents](#c)

---

## 1. Reading in ROOT files <a name="1."></a>

### Higgs Sample, $H \to 4l$ (MC)

As usual, let's start by reading in our files and extracting the TTree `mini`. 

To be certain that the signal events we use to train our neural net are all the process $H \rightarrow 4l$, we will be using Monte-Carlo simulations.

We can also see how many signal events are contained in our file by printing out the length of an random branch, in this case, `runNumber`.

In [ ]:
higgs = uproot.open("http://opendata.cern.ch/eos/opendata/atlas/OutreachDatasets/2020-08-19/4lep/MC/mc_345060.ggH125_ZZ4lep.4lep.root")
higgsTree = higgs["mini"]

numDataEntries = len(higgsTree["runNumber"].array())
print("Tree contains", numDataEntries, "entries")

This file comes from the [DOI](https://opendata.cern/record/15005) `"10.7483/OPENDATA.ATLAS.2Y1T.TLGL"`, which contains events with at least four leptons. If you would like to explore more data files, use the `print_files()` function. 

### $4l$ background (MC)

For our background events, where four leptons are produced by processes other than the Higgs, we will also be using MC simulation, so that we can be certain there are no signal events mixed in when we use them to train our neural network.

In [ ]:
bkg = uproot.open("http://opendata.cern.ch/eos/opendata/atlas/OutreachDatasets/2020-08-19/4lep/MC/mc_363490.llll.4lep.root")
bkgTree = bkg["mini"]

numMCEntries = len(bkgTree["runNumber"].array())
print("Tree contains", numMCEntries, "entries") 

This file also comes from the [DOI](https://opendata.cern/record/15005) `"10.7483/OPENDATA.ATLAS.2Y1T.TLGL"`, which contains events with at least four leptons. If you would like to explore more data files, use the `print_files()` function. 

[Return to contents](#c)

---

## 2. Selecting events and preparing our data  <a name="2."></a>

As before, we'll break our code up into separate __functions__ to make it more manageable and readable.

### 1) Rescaling the simulated events

When MC simulation is compared to data, the contribution of each simulated event needs to be scaled (__reweighted__) to account for differences in how some objects behave in simulation vs in data, as well as the fact that there are different numbers of events in the MC tree than in the data tree.

We will implement this reweighting in the `mcWeights()` function below.

In [ ]:
def mcWeights(data,lumi=10):
    """
    When MC simulation is compared to data the contribution of each simulated event needs to be
    scaled ('reweighted') to account for differences in how some objects behave in simulation
    vs in data, as well as the fact that there are different numbers of events in the MC tree than 
    in the data tree.
    
    Parameters
    ----------
    data : dataframe containing the data extracted from the TTree
    """

    XSection = data["XSection"]
    SumWeights = data["SumWeights"]
    #These values don't change from event to event
    norm = lumi*(XSection*1000)/SumWeights
    
    scaleFactor_ELE = data["scaleFactor_ELE"]
    scaleFactor_MUON = data["scaleFactor_MUON"]
    scaleFactor_LepTRIGGER = data["scaleFactor_LepTRIGGER"]
    scaleFactor_PILEUP = data["scaleFactor_PILEUP"]
    mcWeight = data["mcWeight"]
    #These values do change from event to event
    scale_factors = scaleFactor_ELE*scaleFactor_MUON*scaleFactor_LepTRIGGER*scaleFactor_PILEUP*mcWeight
    
    weight = norm*scale_factors
    return weight

### 2) Selecting events

In our signal, Higgs boson decays into two $Z$ bosons, which each decaying into $2l$ giving 4 in total. We look for events where the $Z$ boson decays into either $e^+e^-$ or $\mu^+\mu^-$. So in this context, when we say `leptons' we are only referring to electrons and muons.

We only have 3 possibilities: $4l = e^+e^- \mu^+ \mu^-$ or $4l = e^+e^-e^+e^-$ or $4l = \mu^+ \mu^-\mu^+ \mu^-$, which all have the same number of positive and negative charges.

Therefore, we want to remove (__cut__) all the events that don't sum up to zero charge, or don't have the right lepton flavours ($e$ or $\mu$). We define these cuts in separate functions, `cut_lep_charge()` and `cut_lep_type()` repectively.

In [ ]:
# A cut on lepton charge
def cut_lep_charge(lep_charge):
    """
    Throw away events where the sum of lepton charges is not equal to 0. The first lepton is [0], 2nd lepton is [1] etc.
    
    Parameters
    ----------
    lep_charge : a list containing the charges of the 4 leptons
    """
    return lep_charge[0] + lep_charge[1] + lep_charge[2] + lep_charge[3] != 0

# Cut on lepton type
def cut_lep_type(lep_type):
    """
    Throw away events where we don't have any of: eeee, mumumumu, eemumu
    
    Electron (and positron) lep_type is 11
    Muon (and anti-muon) lep_type is 13

    Parameters
    ----------
    lep_type : a list containing the types of the 4 leptons
    """
    sum_lep_type = lep_type[0] + lep_type[1] + lep_type[2] + lep_type[3]
    
    return (sum_lep_type != 44) and (sum_lep_type != 48) and (sum_lep_type != 52)

### 3) Defining A New Quantity

When it comes to separating $ZZ$ to $H \to 4l$ events from others, the silver bullet is the __invariant mass__ of the four lepton system.

For two particles, the invariant mass $m$ is calculated from their total energy and momentum:

$m^2 = (E_1 + E_2)^2 - |\vec{p}_1 + \vec{p}_2|^2$

This is a form of Einstein’s famous equation $E^2 = p^2 + m^2$ (rearranged, and dropping our 'natural units', this becomes the familiar $E = mc^2$ for particles at rest).

You can do this for **any number** of particles — just sum their energies and momenta, and plug into this formula.


<div class="alert alert-info">
    
You'll notice we are now using code objects called `dataframes` from the `pandas` library - this is simply a table of data like a spreadsheet. 

In previous notebooks we have looped over every event to process them one by one, now we will use a faster __columnar__ approach. 

For example, if we have a dataframe (table) called `data` with a column of values labelled $y$ (`data['y']`) and another column labelled $x$ (`data['x']`) then we can calculate a third column $z=x \times y$ with `data['z']=data['x'] * data['y']`</div>

In [ ]:
def compute_m4l(data):
    """
    A function to calculate the invariant mass of the four-lepton system using each lepton's momentum, direction and energy.

    Parameters
    ----------
    data : a dataframe containing the kinematics of the 4 leptons
    """
    px_total = np.zeros(len(data)) 
    py_total = np.zeros(len(data))
    pz_total = np.zeros(len(data))
    E_total = np.zeros(len(data))

    for i in range(1, 5): # We loop over each lepton in the event. (Note however, pt is a column of the table data which has one event per row)
        pt = data[f'lep_pt_{i}'].values
        eta = data[f'lep_eta_{i}'].values
        phi = data[f'lep_phi_{i}'].values
        E = data[f'lep_E_{i}'].values
        
        px = pt * np.cos(phi) # Therefore, we calculate px for every event in the column with this single command - no for looping over events required!
        py = pt * np.sin(phi)
        pz = pt * np.sinh(eta)

        px_total += px
        py_total += py
        pz_total += pz
        E_total += E

    m4l_squared = E_total**2 - (px_total**2 + py_total**2 + pz_total**2)
    m4l_squared = np.where(m4l_squared < 0, 0, m4l_squared) # This avoids square rooting a negative (in case Python confuses a small number with a negative)
    m4l = np.sqrt(m4l_squared)
    return m4l


### 4) Prepare our data

This function is very similar to in Notebook 7, but we are now using dataframes (see the blue box above) so we no longer need to loop over all our events.

In [ ]:
def prepareData(tree, sample, fraction):
    """
    A function to create correctly-weighted events with 4 leptons of the correct type and charge. We also want to calculate all 
    our features of interest and add them to the dataframe. Finally, we would like to return all the inputs we'll need for training our 
    neural net (data arrays, features, labels etc). 
    
    Parameters
    ----------
    tree : TTree entry for this event
    sample: name of the data
    fraction: the fraction of events to process from the tree
    """

    data_all = pd.DataFrame()
    numevents = tree.num_entries

    # This for statement opens the TTree as a dataframe called data
    for data in tree.iterate(['lep_charge', 'lep_type', 'lep_pt', 'lep_n',
                              'lep_eta', 'lep_phi', 'lep_E', 'jet_n','jet_pt',
                              'mcWeight', 'scaleFactor_PILEUP',
                              'scaleFactor_ELE', 'scaleFactor_MUON',
                              'scaleFactor_LepTRIGGER', 'XSection', 'SumWeights'],
                             library="pd",
                             entry_stop=int(numevents * fraction)):

        
        nIn = len(data.index) # The number of events before we apply any cuts
        
        # We now remove any events from the dataframe that fail our cuts:
        # .apply returns a list of True/False on whether we want to cut it.
        # ~fail flips all the True<->False, so the True's become a list of ones we want to keep
        # data[~fail] then only keeps the one's with a True, i.e. passing our cut
        fail = data['lep_charge'].apply(cut_lep_charge)
        data = data[~fail]

        fail =  data['lep_type'].apply(cut_lep_type)
        data = data[~fail]

        if 'data' not in sample: # We need to calculate mc weights for MC data
            data['totalWeight'] = mcWeights(data)
        else:
            data['totalWeight'] = np.full(len(data), 1.0)

        # Extract features for each lepton (1 to 4)
        # Since each event has 4 leptons, data['lep_pt'] has the pt for each lepton in it. To extract the lepton pt for the i-th lepton we have to do 
        # .apply(lambda x: x[i-1]) to get the i-th value.
        # If this is unclear don't worry, it is a small cost for speeding up our code!
        for i in range(1, 5):
            data[f'lep_pt_{i}'] = data['lep_pt'].apply(lambda x: x[i-1])
            data[f'lep_phi_{i}'] = data['lep_phi'].apply(lambda x: x[i-1])
            data[f'lep_eta_{i}'] = data['lep_eta'].apply(lambda x: x[i-1])
            data[f'lep_E_{i}'] = data['lep_E'].apply(lambda x: x[i-1])
            data[f'jet_pt_{i}'] = data['lep_E'].apply(lambda x: x[i-1] if len(x) >= i else 0) # If we don't have 4 jets in the event, we have this 
            # else clause to set the value to 0
        
        nOut = len(data.index)
        
        #print(f"For sample {sample}, Events before cuts: {nIn}, After cuts: {nOut}")
        data_all = pd.concat([data_all, data], ignore_index=True)

    # We can now apply our calculation of the 4-lepton invariant mass
    data_all['m4l'] = compute_m4l(data_all)
    
    # We now choose the features we want to save, lets put them in a list called feature_columns
    feature_columns = []
    for i in range(1, 5):
        feature_columns += [f'lep_pt_{i}', f'lep_phi_{i}',f'lep_eta_{i}',f'lep_E_{i}',f'jet_pt_{i}']
    feature_columns += ['jet_n', 'm4l']

    # We turn our dataframe 'data' into a numpy array like we had for the cats and dogs example
    features = data_all[feature_columns].to_numpy()

    # Then we need to create our labels depending on what type of sample it is.
    if sample == 'Higgs':
        label = 1
    else:
        label = 0
    labels = np.full(len(features), label)

    return features, data_all['totalWeight'].to_numpy(), labels, feature_columns

Now we've got all our functions, let's prepare the data and then plot what is going into our neural net.

In [ ]:
#Prepare our signal and bakground data using the functions we've defined above
signal, signal_weights, signal_labels, feature_names = prepareData(higgsTree, 'Higgs', fraction=1)
background, background_weights, background_labels, feature_names = prepareData(bkgTree, 'Bkg', fraction=1)

#Cobine and shuffle our neural net inputs, like we did in Notebook 8
features_combined = np.vstack([signal, background])
weights_combined = np.concatenate([signal_weights,background_weights])
labels_combined = np.concatenate([signal_labels,background_labels])

indices = np.random.permutation(len(features_combined))
features = features_combined[indices]
labels = labels_combined[indices]
weights = weights_combined[indices]

Since we are using MC we need to remember to use event weights in our histogramming, rather than the event counts - thankfully this can really easily be done as shown below, using the `weights` parameter of the `hist` function.

In [ ]:
plt.figure(figsize=(14, 15))
for i in range(len(feature_names)):
    combined = np.concatenate([signal[:, i], background[:, i]])
    bins = np.linspace(combined.min(), combined.max(), 100)

    plt.subplot(5, 5, i + 1)

    # Order: background first, then Higgs (so Higgs is on top)
    data = [background[:, i], signal[:, i]]
    plot_weights = [background_weights, signal_weights]
    plot_labels = ['ZZ', 'Higgs']
    colors = ['purple', 'green']

    plt.hist(data, bins=bins, stacked=True, weights=plot_weights, label=plot_labels, color=colors) #Wweights parameter lets us histogram with our MC weights

    plt.title(feature_names[i])
    plt.xlabel(feature_names[i])
    plt.ylabel('Weighted Count')
    plt.xlim(0, combined.max() * 0.3)

plt.tight_layout()
plt.show()


Look at the 4 lepton invariant mass distribution - you may already be able to guess that it could be an important feauture for classifying events. Let's focus on this histogram to make this easier to see.

In [ ]:
plt.figure()
i = len(feature_names) - 1

# Get min and max from combined data to set consistent bins
combined = np.concatenate([signal[:, i], background[:, i]])
bins = np.linspace(combined.min(), combined.max(), 400)

# Reverse the order: background first, then Higgs
data = [background[:, i], signal[:, i]]
plot_weights = [background_weights, signal_weights]
plot_labels = ['ZZ', 'Higgs']
colors = ['purple', 'green']

plt.hist(data, bins=bins, stacked=True, weights=plot_weights, label=plot_labels, color=colors)
plt.title(feature_names[i])
plt.xlabel(feature_names[i])
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.xlim(0, 5e5)
plt.show()


[Return to contents](#c)

---

## 3. Training a Neural Net  <a name="3."></a>

You know what to do here! It is almost identical to what we did in Notebook 8. See if you can use this data to train a neural net, then assess its performance.

<div class="alert alert-info">
    
We need to include the MC event weights when we train. This time split the data like this:
<br>
```python
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    features_scaled, labels, weights, test_size=0.2, random_state=42
)
```
<br>

and then pass the argument `sample_weight=w_train` into `model.fit()`.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# Standardize features
# ... your code here

# Split data
# ... your code here

#Define our network architechture
model = models.Sequential([
# ... your code here
])

#Compile your model
model.compile(
# ... your code here
)

#Train your model
history = model.fit(
# ... your code here
)

#Evaluate the accuracy of your model
# ... your code here

<details>
<summary>Answers: </summary>

```python

import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# Standardize features
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Split data
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    features_scaled, labels, weights, test_size=0.2, random_state=42
)

#Define our network architechture
model = models.Sequential([
    layers.Input(shape=(features.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')  # Binary classification
])

#Compile your model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

#Train your model
history = model.fit(
    X_train, y_train, sample_weight=w_train,
    validation_data=(X_test, y_test, w_test),
    epochs=100,
    batch_size=512,
    verbose=1
)

#Evaluate the accuracy of your model
loss, acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {acc:.4f}")</details>
```

Now, we should plot the accuracy (for both the training and validation datasets) and Classifier Score Distribution to see how well our neural net works. Don't forget to include the weights in the Classifier Score Distribution!

<div class="alert alert-info">
    
If you are having trouble remembering how to create these plots, please refer back to Notebook 8.

In [ ]:
#Plot the training accuracy vs epoch and the validation accuracy vs epoch on the same plot

# ... your code here

<details>
<summary>Answers: </summary>

```python

plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label = 'Validation Accuracy')
plt.xlabel('Epoch (becoming more trained -->)')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.show()

```

</details>

In [ ]:
#Plot the Classifier Score Distribution for both the signal and background using the event weights

# ... your code here

<details>
<summary>Answers: </summary>

```python

y_pred_scores = model.predict(X_test).ravel()  # sigmoid output: values between 0 and 1

# Select weights for test set:
signal_score_weights = w_test[y_test == 1]
background_score_weights = w_test[y_test == 0]

# And select corresponding prediction scores:
signal_scores = y_pred_scores[y_test == 1]
background_scores = y_pred_scores[y_test == 0]

# Plot
plt.figure()
plt.hist(signal_scores, bins=50, alpha=0.6, label='Higgs (Signal)', color='green', weights=signal_score_weights)
plt.hist(background_scores, bins=50, alpha=0.6, label='ZZ (Background)', color='purple', weights=background_score_weights)

plt.xlabel("Classifier Output (score)")
plt.ylabel("Normalized Count")
plt.title("Classifier Score Distribution")
plt.legend()
plt.tight_layout()
plt.yscale('log')
plt.show()
```
</details>

[Return to contents](#c)

---


## 4. Running on ATLAS data  <a name="4."></a>

So far, we have trained and validated our neural network using MC simulation only!

To use our network to try to identify $H\rightarrow 4l$  events ib real ATLAS collision data, we take a similar approach to before:

1. Open the data files
   - Just as we did for the MC, use the [DOI](https://opendata.cern/record/15005) for the 4 lepton dataset, `"10.7483/OPENDATA.ATLAS.2Y1T.TLGL"`, and the `print_files()` function to find the files.
   - Then you'll have to combine the features of each with `np.vstack(data_features_all)` where `data_features_all = [features1, features2, ...]`)

2. Prepare the data files' `data_features, _ , feature_names = prepareData(higgsTree, 'Higgs', fraction=1)`

In [ ]:
# ... your code here for points 1 and 2

<details>
<summary>Answers: </summary>

```python

data_features_all = []
for let in ['A', 'B', 'C', 'D']:
    dataFile = uproot.open(f"http://opendata.cern.ch/eos/opendata/atlas/OutreachDatasets/2020-08-19/4lep/Data/data_{let}.4lep.root")
    dataTree = dataFile['mini']
    data_features, _ , _ , feature_names = prepareData(dataTree, 'data', fraction=1)
    data_features_all.append(data_features)
data_features = np.vstack(data_features_all)
```
</details>

3. Get a feel for the properties of the data by plotting it on top of the MC - is there a peak in data where the MC suggests we might find the Higgs boson? What do you notice about the data compared to the MC?

In [ ]:
# ... your code here for point 3

<details>
<summary>Answers: </summary>

```python

plt.figure()
i = len(feature_names) - 1

# Set consistent binning
combined = np.concatenate([data_features[:, i], signal[:, i], background[:, i]])
bins = np.linspace(combined.min(), combined.max(), 600)
bin_centers = 0.5 * (bins[1:] + bins[:-1])

# Stack MC: background first, then signal (so signal is on top)
mc_data = [background[:, i], signal[:, i]]
mc_weights = [background_weights, signal_weights]
mc_labels = ['ZZ', 'Higgs']
mc_colors = ['purple', 'green']

# Plot stacked MC histograms
plt.hist(mc_data, bins=bins, weights=mc_weights, stacked=True,
         label=mc_labels, color=mc_colors, alpha=0.6)

# Plot ATLAS data as black points with sqrt(N) error bars
data_counts, _ = np.histogram(data_features[:, i], bins=bins)
data_errors = np.sqrt(data_counts)
plt.errorbar(bin_centers, data_counts, yerr=data_errors, fmt='o', color='black', label='ATLAS')

# Formatting
plt.title(feature_names[i])
plt.xlabel(feature_names[i])
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.xlim(0, 3e5)
plt.show()
```
</details>

<details>
<summary>Answers: </summary>

You should notice that there is a huge gap between the MC and the data - this might suggest that we haven't got MC files for enough background processes in here yet!

</details>

4. We have already trained our neural net to identify Higgs signal events using our labelled MC - now let's use it to identify Higgs events in data, where we do not have access to the 'truth' information. To do this, run the neural net on the data as `y_scores = model.predict(scaled_data_features).ravel()`
   
5) Plot the Classifier Score Distribution - how can we use this to tell if our data contains any Higgs events?

In [ ]:
# ... your code here for points 4 and 5

<details>
<summary>Answers: </summary>

```python

scaler = StandardScaler()
scaled_data_features = scaler.fit_transform(data_features)
y_scores = model.predict(scaled_data_features).ravel()  # sigmoid output: values between 0 and 1

# Plot the score distribution
plt.figure()
plt.hist(y_scores, bins=50, color='blue', alpha=0.7, label='ATLAS data', density=True)

plt.xlabel("Classifier Output (score)")
plt.ylabel("Normalized Count")
plt.title("Classifier Score on ATLAS Data")
plt.legend()
plt.yscale('log')
plt.tight_layout()
plt.show()
```

</details>

<details>
<summary>Answers: </summary>

Hopefully you found some events with high classifier scores, which we might be convinced are Higgs events. Generally you should notice that we need more data!

</details>

[Return to contents](#c)

---

## Optional extra exercises / 'Do your own project' ideas  <a name="6."></a>

You may have noticed some flaws in this analysis - can you improve this analysis?

For example:
1. We have only included one Higgs file and one background file, we need to include the rest to get a good match between data and MC.
2. Can you optimise the Neural Net further?
3. Perhaps by looking at https://atlas.web.cern.ch/Atlas/GROUPS/PHYSICS/PAPERS/HIGG-2018-29/ you can discover some new cuts on the leptons we could use, e.g. put constraints on the minimum pt, to get cleaner data. Or, maybe you can find some new variables we can calculate to train the Neural Net on.

In [ ]:
# For each new variable we would add a new functions e.g.

def invariant_mass(p1, p2):
    E = p1['E'] + p2['E']
    px = p1['pt']*np.cos(p1['phi']) + p2['pt']*np.cos(p2['phi'])
    py = p1['pt']*np.sin(p1['phi']) + p2['pt']*np.sin(p2['phi'])
    pz = p1['pt']*np.sinh(p1['eta']) + p2['pt']*np.sinh(p2['eta'])
    return np.sqrt(E**2 - (px**2 + py**2 + pz**2))

def compute_best_Z_pairing(df):
    Z_mass = 91.2  # GeV
    mZ1_all = []
    mZ2_all = []

    for i in range(len(df)):
        lep = {}
        for j in range(1, 5):
            lep[j] = {
                'pt': df[f'lep_pt_{j}'].iloc[i],
                'eta': df[f'lep_eta_{j}'].iloc[i],
                'phi': df[f'lep_phi_{j}'].iloc[i],
                'E': df[f'lep_E_{j}'].iloc[i]
            }

        pairings = [
            ((1, 2), (3, 4)),
            ((1, 3), (2, 4)),
            ((1, 4), (2, 3))
        ]

        best_delta = float('inf')
        best_mZ1 = None
        best_mZ2 = None

        for (a1, a2), (b1, b2) in pairings:
            m1 = invariant_mass(lep[a1], lep[a2])
            m2 = invariant_mass(lep[b1], lep[b2])
            delta = abs(m1 - Z_mass)
            if delta < best_delta:
                best_delta = delta
                best_mZ1 = m1
                best_mZ2 = m2

        mZ1_all.append(best_mZ1)
        mZ2_all.append(best_mZ2)

    return np.array(mZ1_all), np.array(mZ2_all)

# In the read data function we need to add:
# data_all['mZ1'], data_all['mZ2'] = compute_best_Z_pairing(data_all) 
# Then add 'mZ1' and 'mZ2' to feature_columns

# Another possible variable is:

def delta_r(eta1, phi1, eta2, phi2):
    dphi = np.abs(phi1 - phi2)
    dphi = np.where(dphi > np.pi, 2*np.pi - dphi, dphi)
    deta = eta1 - eta2
    return np.sqrt(deta**2 + dphi**2)

[Return to contents](#c)

---

<div class="alert alert-success">

__Congratulations!__ You've successfully completed ALL the notebooks! Very few make it this far, you should be proud of yourself! 

It's time to take your new scientific skills solo and start doing your own research - use the suggestions above, or any of the Extra exercises / 'Do your own project' prompts from the notebooks as inspiration.

In addition to the samples used in these notebooks, there is much more data available! We have used the <a href="https://opendata.atlas.cern/docs/documentation/example_analyses/analysis_examples_education_2020"> "2020 13 TeV"</a> data in these examples, but there is also the newer, much bigger <a href="https://opendata.atlas.cern/docs/data/for_education/13TeV25_details"> "2025 13 TeV"</a> sample, which you are encouraged to explore!
    
Well done!
</div>